# DAYA - Visualizations
## Generate Professional Charts and Dashboards

This notebook creates 9 professional visualizations:
1. Bed occupancy trend
2. Distribution histogram
3. Box plot
4. Monthly pattern
5. Weekly pattern
6. Forecast comparison
7. Alert zones
8. Summary dashboard
9. Statistics table

## Setup

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✅ Libraries loaded!")

## Load Processed Data

In [ ]:
# Load data
daily_beds = pd.read_csv('data/processed/daily_bed_occupancy.csv')
daily_beds['date'] = pd.to_datetime(daily_beds['date'])

with open('data/processed/analysis_summary.json', 'r') as f:
    summary = json.load(f)

print(f"✓ Loaded {len(daily_beds)} days of data")
print(f"✓ Loaded analysis summary")

# Create output directory
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

## 1. Bed Occupancy Trend

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(daily_beds['date'], daily_beds['bed_count'], 
        linewidth=2, color='#2E86AB', label='Daily Bed Count', marker='o', markersize=4)
ax.plot(daily_beds['date'], daily_beds['MA_7'], 
        linewidth=2, color='#F18F01', label='7-Day Moving Average', linestyle='--')
ax.plot(daily_beds['date'], daily_beds['MA_14'], 
        linewidth=2, color='#C73E1D', label='14-Day Moving Average', linestyle='--')

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Bed Count', fontsize=12, fontweight='bold')
ax.set_title('Hospital Bed Occupancy Trend with Moving Averages', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / '01_bed_occupancy_trend.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 01_bed_occupancy_trend.png")

## 2. Distribution Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(daily_beds['bed_count'], bins=20, edgecolor='black', 
        color='#06A77D', alpha=0.7)
ax.axvline(daily_beds['bed_count'].mean(), color='red', 
           linestyle='--', linewidth=2, label=f'Mean: {daily_beds["bed_count"].mean():.0f}')
ax.axvline(daily_beds['bed_count'].median(), color='orange', 
           linestyle='--', linewidth=2, label=f'Median: {daily_beds["bed_count"].median():.0f}')

ax.set_xlabel('Bed Count', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Daily Bed Occupancy', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / '02_bed_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 02_bed_distribution.png")

## 3. Monthly Pattern

In [ ]:
monthly_avg = daily_beds.groupby('month')['bed_count'].agg(['mean', 'std'])
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(14, 6))

x = range(len(monthly_avg))
ax.bar(x, monthly_avg['mean'], yerr=monthly_avg['std'], 
       color='#F18F01', alpha=0.7, edgecolor='black', capsize=5)
ax.set_xticks(x)
ax.set_xticklabels([months[i-1] for i in monthly_avg.index])
ax.set_xlabel('Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Bed Count', fontsize=12, fontweight='bold')
ax.set_title('Monthly Average Bed Occupancy (with Std Dev)', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / '04_monthly_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 04_monthly_pattern.png")

## 4. Weekly Pattern

In [ ]:
weekly_avg = daily_beds.groupby('day_of_week')['bed_count'].mean()
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#2E86AB' if i < 5 else '#C73E1D' for i in range(7)]
ax.bar(range(7), weekly_avg, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(7))
ax.set_xticklabels(days, rotation=45, ha='right')
ax.set_xlabel('Day of Week', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Bed Count', fontsize=12, fontweight='bold')
ax.set_title('Weekly Pattern: Average Bed Occupancy by Day', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2E86AB', alpha=0.7, label='Weekday'),
                   Patch(facecolor='#C73E1D', alpha=0.7, label='Weekend')]
ax.legend(handles=legend_elements, loc='best')

plt.tight_layout()
plt.savefig(output_dir / '05_weekly_pattern.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 05_weekly_pattern.png")

## 5. Forecast Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# Historical data
ax.plot(daily_beds['date'], daily_beds['bed_count'], 
        linewidth=2, marker='o', label='Actual', color='#2E86AB', markersize=4)

# Forecast dates
last_date = daily_beds['date'].max()
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=7)

# MA forecast
ma_forecast = [summary['forecasts']['ma_7day']] * 7
ax.plot(forecast_dates, ma_forecast, 
        linewidth=2, marker='s', label='MA Forecast', 
        color='#F18F01', linestyle='--', markersize=6)

# Trend forecast
trend_forecast = summary['forecasts']['trend_7day']
ax.plot(forecast_dates, trend_forecast, 
        linewidth=2, marker='^', label='Trend Forecast', 
        color='#C73E1D', linestyle='--', markersize=6)

ax.axvline(x=last_date, color='gray', linestyle=':', linewidth=2, label='Forecast Start')

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Bed Count', fontsize=12, fontweight='bold')
ax.set_title('7-Day Bed Occupancy Forecast (MA vs Trend)', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / '06_forecast_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 06_forecast_comparison.png")

## 6. Alert Zones

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

warning = summary['alerts']['warning_threshold']
critical = summary['alerts']['critical_threshold']

ax.plot(daily_beds['date'], daily_beds['bed_count'], 
        linewidth=2, color='#2E86AB', label='Daily Bed Count', marker='o', markersize=4)

# Alert zones
ax.axhspan(0, warning, alpha=0.1, color='green', label='Normal Zone')
ax.axhspan(warning, critical, alpha=0.1, color='orange', label='Warning Zone')
ax.axhspan(critical, daily_beds['bed_count'].max() + 50, alpha=0.1, color='red', label='Critical Zone')

ax.axhline(y=warning, color='orange', linestyle='--', linewidth=2, label=f'Warning: {warning:.0f}')
ax.axhline(y=critical, color='red', linestyle='--', linewidth=2, label=f'Critical: {critical:.0f}')

ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Bed Count', fontsize=12, fontweight='bold')
ax.set_title('Bed Occupancy with Alert Thresholds', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=10)
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(output_dir / '07_alert_zones.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 07_alert_zones.png")

## Summary

In [ ]:
print("="*70)
print("✓ ALL VISUALIZATIONS GENERATED!")
print("="*70)

print(f"\n📊 Generated visualizations:")
print(f"  1. Bed occupancy trend with moving averages")
print(f"  2. Bed count distribution histogram")
print(f"  3. Monthly pattern analysis")
print(f"  4. Weekly pattern analysis")
print(f"  5. 7-day forecast comparison")
print(f"  6. Alert zones visualization")

print(f"\n📁 All files saved to: {output_dir.absolute()}")
print("="*70)